# protosearch — Protein Family Survey

Generic template for surveying any protein family across a user-supplied proteome.

**Pipeline:** HMMER retrieval → ESM2 embedding → t-SNE clustering → IQ-TREE phylogeny → Ancestral reconstruction → Cluster labeling

---
**To use:** Fill in cells marked `USER INPUT` and run top to bottom.

For a worked example with AGPAT acyltransferases see `agpat_crustacea.ipynb` in the agpat/ directory.

In [ ]:
# [00] Clone repo and configure path
import subprocess, sys
subprocess.run(['git', 'clone', 'https://github.com/bjdraper/protosearch.git'], check=True)
sys.path.insert(0, 'protosearch')

In [ ]:
# [02] Install dependencies
!apt-get install -qq hmmer mafft fasttree
# IQ-TREE2 is not in apt — download binary from GitHub releases
!wget -q https://github.com/iqtree/iqtree2/releases/download/v2.3.6/iqtree-2.3.6-Linux-intel.tar.gz \
  && tar -xzf iqtree-2.3.6-Linux-intel.tar.gz \
  && cp iqtree-2.3.6-Linux-intel/bin/iqtree2 /usr/local/bin/iqtree2 \
  && chmod +x /usr/local/bin/iqtree2
!pip install -q fair-esm faiss-cpu leidenalg python-igraph openTSNE biopython logomaker pyyaml tqdm seaborn

In [ ]:
# [02b] Embedding backend  ── USER INPUT ──────────────────────────────────────
# Option A — NVIDIA NIM (recommended for Colab: no 2.5 GB download, no GPU needed):
#   1. Get a free key at https://build.nvidia.com/settings/api-keys
#   2. Paste it below — EMBED_BACKEND is set automatically.
#   Free tier: 40 RPM.  At batch_size=32, API calls take ~4s each → actual RPM
#   stays ~15, well under the cap.  NVIDIA_RPM=120 gives 0.6s sleep (safe).
#   Retry on 429 is handled automatically with exponential backoff.
#
# Option B — Local ESM2 (needs Colab GPU runtime, downloads ~2.5 GB on first run):
#   Leave NVIDIA_API_KEY = '' — EMBED_BACKEND falls back to 'local'.

NVIDIA_API_KEY    = ''     # paste nvapi-... key here
NVIDIA_BATCH_SIZE = 32     # sequences per API call (free tier: RPM limit is on requests, not seqs)
NVIDIA_RPM        = 120    # used only to set inter-batch sleep (60/120+0.1 ≈ 0.6s); actual RPM ~15

EMBED_BACKEND = 'nvidia' if NVIDIA_API_KEY else 'local'
print(f'Embedding backend: {EMBED_BACKEND}')
if EMBED_BACKEND == 'nvidia':
    print(f'  batch_size={NVIDIA_BATCH_SIZE}, rpm_limit={NVIDIA_RPM}')
    print(f'  inter-batch sleep ≈ {60/NVIDIA_RPM + 0.1:.1f}s  |  retries on 429 with backoff')
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# [03] Mount Google Drive and set survey name
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

# ── USER INPUT ────────────────────────────────────────────────────────────────
SURVEY_NAME = 'agpat_crustacea'   # must match the folder name in MyDrive/
# ─────────────────────────────────────────────────────────────────────────────

PROJECT_ROOT = Path(f'/content/drive/MyDrive/{SURVEY_NAME}')
DATA_DIR     = PROJECT_ROOT / 'data'
RESULTS_DIR  = PROJECT_ROOT / 'results'
for d in [DATA_DIR, RESULTS_DIR]: d.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

In [ ]:
# [04] Define protein family  ── USER INPUT ────────────────────────────────────
# AGPAT acyltransferase family — Crustacean survey (172 genomes + 178 transcriptomes)

REFERENCE_PROBES = [
    {'id': 'AGPAT1_HUMAN',   'acc': 'Q99943', 'label': 'AGPAT1',             'colour': '#D94040'},
    {'id': 'AGPAT2_HUMAN',   'acc': 'O15120', 'label': 'AGPAT2',             'colour': '#E05C5C'},
    {'id': 'AGPAT3_HUMAN',   'acc': 'Q9NRZ7', 'label': 'AGPAT3',             'colour': '#E87878'},
    {'id': 'AGPAT4_HUMAN',   'acc': 'Q9NRZ5', 'label': 'AGPAT4',             'colour': '#F09090'},
    {'id': 'AGPAT5_HUMAN',   'acc': 'Q9NUQ2', 'label': 'AGPAT5',             'colour': '#F8B0B0'},
    {'id': 'LCLAT1_HUMAN',   'acc': 'Q6UWP7', 'label': 'LCLAT1',             'colour': '#F57F17'},
    {'id': 'LCLAT1_MOUSE',   'acc': 'Q6NVG1', 'label': 'LCLAT1 (mouse)',     'colour': '#E65100'},
    {'id': 'LCLAT1_DANRE',   'acc': 'Q6NYV8', 'label': 'LCLAT1 (zebrafish)', 'colour': '#FB8C00'},
    {'id': 'GPAT4_HUMAN',    'acc': 'Q86UL3', 'label': 'GPAT4',              'colour': '#00695C'},
    {'id': 'GPAT3_HUMAN',    'acc': 'Q53EU6', 'label': 'GPAT3',              'colour': '#2E7D32'},
    {'id': 'GPAT1_MOUSE',    'acc': 'Q61586', 'label': 'GPAT1 (mouse)',      'colour': '#004D40'},
    {'id': 'GPAT2_MOUSE',    'acc': 'Q8BYM8', 'label': 'GPAT2 (mouse)',      'colour': '#1B5E20'},
    {'id': 'AGPAT2_MOUSE',   'acc': 'Q8K3K7', 'label': 'AGPAT2 (mouse)',     'colour': '#B71C1C'},
    {'id': 'LPCAT2_HUMAN',   'acc': 'Q7L5N7', 'label': 'LPCAT2',             'colour': '#1565C0'},
    {'id': 'PlsC_ECOLI',     'acc': 'P26647', 'label': 'PlsC (E.coli)',      'colour': '#388E3C'},
    {'id': 'PlsC_BACSU',     'acc': 'O34478', 'label': 'PlsC (B.sub)',       'colour': '#33691E'},
    {'id': 'PlsC_THAPS',     'acc': 'B8BZR1', 'label': 'PlsC (diatom)',      'colour': '#558B2F'},
    {'id': 'Tafazzin_HUMAN', 'acc': 'Q16635', 'label': 'Tafazzin [ctrl]',    'colour': '#4A148C'},
    {'id': 'LPCAT3_HUMAN',   'acc': 'Q6P1A2', 'label': 'LPCAT3 [ctrl]',      'colour': '#B0BEC5'},
    {'id': 'MBOAT7_HUMAN',   'acc': 'Q96N66', 'label': 'MBOAT7 [ctrl]',      'colour': '#90A4AE'},
]

# Pfam domain ID for HMMER prefilter
PFAM_IDS = ['PF01553']   # acyltransferase 3 domain

# Protein length filter
MIN_LEN = 200
MAX_LEN = 500

# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# [05] Write config.yaml
import yaml
config = {
    'paths': {'data_dir': str(DATA_DIR), 'results_dir': str(RESULTS_DIR)},
    'reference_probes': REFERENCE_PROBES,
    'pfam_ids': PFAM_IDS,
    'length_filter': {'min': MIN_LEN, 'max': MAX_LEN},
    'embedding': {'model': 'esm2_t33_650M_UR50D', 'layer': 33, 'batch_size': 32},
    'clustering': {'leiden_resolution': 2.0, 'tsne_perplexity': 100},
    'tree': {'aligner': 'mafft', 'model': 'LG+G4'},
}
config_path = PROJECT_ROOT / 'config.yaml'
with open(config_path, 'w') as f: yaml.dump(config, f)
print('Config written to', config_path)

In [ ]:
# [06] Download reference probes from UniProt
from protosearch import utils
probe_fasta = DATA_DIR / 'reference_probes.faa'
utils.fetch_uniprot_sequences([p['acc'] for p in REFERENCE_PROBES], probe_fasta)
print(f'Fetched {len(REFERENCE_PROBES)} reference probes → {probe_fasta}')

In [ ]:
# [07] Download Pfam HMM profile(s)
from protosearch import search
hmm_dir = DATA_DIR / 'hmm_profiles'
hmm_dir.mkdir(exist_ok=True)
for pfam_id in PFAM_IDS:
    search.download_pfam_hmm(pfam_id, hmm_dir)
print('HMM profiles ready in', hmm_dir)

In [ ]:
# [08] Scan Drive for proteome FASTA files  ── USER INPUT ─────────────────────
import os
proteins_raw_dir = PROJECT_ROOT / 'input_proteomes'
found = sorted(proteins_raw_dir.glob('*.faa')) + sorted(proteins_raw_dir.glob('*.fasta'))
print(f'Found {len(found)} FASTA files in {proteins_raw_dir}')
for f in found[:5]: print(' ', f.name)
if len(found) > 5: print(f'  ... and {len(found) - 5} more')

# Set INPUT_FASTAS to the files you want to include
INPUT_FASTAS = found   # or list a subset: [found[0], found[2]]
# ─────────────────────────────────────────────────────────────────────────────

raw_fasta = DATA_DIR / 'proteins_raw_combined.faa'
with open(raw_fasta, 'w') as out:
    for f in INPUT_FASTAS:
        out.write(open(f).read())
print(f'Combined {len(INPUT_FASTAS)} file(s) → {raw_fasta}')

In [ ]:
# [09] Filter proteins by length + dedup
from protosearch import utils
filtered_fasta = DATA_DIR / 'proteins_filtered.faa'
n = utils.filter_and_dedup(raw_fasta, filtered_fasta, min_len=MIN_LEN, max_len=MAX_LEN)
print(f'{n} proteins after length filter ({MIN_LEN}–{MAX_LEN} AA) → {filtered_fasta}')

In [ ]:
# [10] HMMER prefilter
from protosearch import search
hmmer_hits_fasta = DATA_DIR / 'hmmer_hits.faa'
hmmer_hits_tsv   = RESULTS_DIR / 'hmmer_hits.tsv'
search.run_hmmer(filtered_fasta, hmm_dir, hmmer_hits_fasta, hmmer_hits_tsv)
print(f'HMMER hits → {hmmer_hits_fasta}')

In [ ]:
# [11] ESM2 embedding of HMMER hits
from protosearch import embed
hits_emb_path = DATA_DIR / 'embeddings' / 'hmmer_hits.npy'
hits_ids_path = DATA_DIR / 'embeddings' / 'hmmer_hits_ids.txt'
_bs = NVIDIA_BATCH_SIZE if EMBED_BACKEND == 'nvidia' else 32
embed.embed_fasta(hmmer_hits_fasta, hits_emb_path, hits_ids_path,
                  backend=EMBED_BACKEND, api_key=NVIDIA_API_KEY,
                  batch_size=_bs, rpm_limit=NVIDIA_RPM)
print('Embeddings saved.')

In [ ]:
# [12] ESM2 embedding of reference probes
ref_emb_path = DATA_DIR / 'embeddings' / 'ref_embeddings.npy'
ref_ids_path = DATA_DIR / 'embeddings' / 'ref_embeddings_ids.txt'
_bs = NVIDIA_BATCH_SIZE if EMBED_BACKEND == 'nvidia' else 32
embed.embed_fasta(probe_fasta, ref_emb_path, ref_ids_path,
                  backend=EMBED_BACKEND, api_key=NVIDIA_API_KEY,
                  batch_size=_bs, rpm_limit=NVIDIA_RPM)
print('Reference embeddings saved.')

In [ ]:
# [16] Load embeddings for clustering
import numpy as np
emb  = np.load(hits_emb_path)
ids  = open(hits_ids_path).read().splitlines()
ref_emb = np.load(ref_emb_path)
ref_ids = open(ref_ids_path).read().splitlines()
print(f'Hits: {emb.shape}, Refs: {ref_emb.shape}')

In [ ]:
# [17] Leiden clustering
# resolution: lower = fewer clusters. 0.5 for small datasets (<100 hits), 2.0 for large.
from protosearch import cluster
labels = cluster.leiden_cluster(emb, resolution=2.0)
print(f'{len(set(labels))} clusters found')

In [ ]:
# [18] t-SNE plot
# perplexity must be < N/3 where N = number of HMMER hits.
# Use 8 for small datasets (~25 hits), 30–100 for large datasets.
from protosearch import cluster, visualize
tsne_coords = cluster.run_tsne(emb, perplexity=100)
visualize.plot_tsne(tsne_coords, labels, ids, ref_emb, ref_ids,
                    REFERENCE_PROBES, RESULTS_DIR / 'tsne.png')

In [ ]:
# [20] MAFFT + FastTree per cluster
from protosearch import tree, utils
for clu in sorted(set(labels)):
    clu_ids = [i for i, l in zip(ids, labels) if l == clu]
    clu_fasta = RESULTS_DIR / f'cluster_{clu}_hits.faa'
    utils.extract_sequences(hmmer_hits_fasta, clu_ids, clu_fasta)
    tree.align_and_tree(clu_fasta, RESULTS_DIR / f'cluster_{clu}')
print('Trees built.')

In [ ]:
# [21] IQ-TREE2 ancestral state reconstruction
from protosearch import tree
for clu in sorted(set(labels)):
    aligned = RESULTS_DIR / f'cluster_{clu}_aligned.faa'
    if aligned.exists():
        tree.run_iqtree_asr(aligned, RESULTS_DIR / f'cluster_{clu}_iqtree')
print('ASR complete.')

In [ ]:
# [22] Parse ASR results + generate sequence logos
from protosearch import asr, visualize
for clu in sorted(set(labels)):
    state_file = RESULTS_DIR / f'cluster_{clu}_iqtree' / 'alignment.state'
    if state_file.exists():
        anc = asr.parse_state_file(state_file)
        visualize.plot_ancestral_logo(anc, RESULTS_DIR / f'cluster_{clu}_ancestral_logo.png')
print('Done.')